<a href="https://colab.research.google.com/github/mitalidaduria/nlp-payments-lab/blob/main/MLflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install -q mlflow xgboost optuna scikit-learn pandas joblib pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 796.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [12]:
import os
os.makedirs("scripts", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)

with open("scripts/train_with_tracking.py", "w") as f:
    f.write('''import os
import joblib
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import mlflow
import mlflow.xgboost
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, auc, f1_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def create_synthetic_data(n_samples=10000, fraud_rate=0.02, n_features=20, random_state=42):
    weights = [1 - fraud_rate, fraud_rate]
    X_raw, y_raw = make_classification(
        n_samples=n_samples, n_features=n_features, n_informative=10,
        n_redundant=5, weights=weights, random_state=random_state
    )
    df = pd.DataFrame(X_raw, columns=[f"feature_{i}" for i in range(n_features)])
    df['is_fraud'] = y_raw
    return df

def main():
    DATA_CONFIG = {"n_samples": 10000, "fraud_rate": 0.02, "n_features": 20, "random_state": 42}
    mlflow.set_experiment("payment-fraud-detection")

    with mlflow.start_run(run_name="xgb_optuna_50trials") as run:
        print(f"Started MLflow Run ID: {run.info.run_id}")
        mlflow.log_params({f"data_{k}": v for k, v in DATA_CONFIG.items()})

        df = create_synthetic_data(**DATA_CONFIG)
        X = df.drop(columns=["is_fraud"])
        y = df["is_fraud"]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=DATA_CONFIG["random_state"]
        )

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        joblib.dump(scaler, "artifacts/feature_pipeline.pkl")
        mlflow.log_artifact("artifacts/feature_pipeline.pkl", artifact_path="pipeline")

        def objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 200),
                'max_depth': trial.suggest_int('max_depth', 3, 9),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'scale_pos_weight': (1 - DATA_CONFIG["fraud_rate"]) / DATA_CONFIG["fraud_rate"],
                'random_state': DATA_CONFIG["random_state"],
                'eval_metric': 'aucpr'
            }
            cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=DATA_CONFIG["random_state"])
            pr_aucs = []
            for train_idx, val_idx in cv.split(X_train_scaled, y_train):
                X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
                y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
                model = xgb.XGBClassifier(**params)
                model.fit(X_tr, y_tr)
                preds_probs = model.predict_proba(X_val)[:, 1]
                precision, recall, _ = precision_recall_curve(y_val, preds_probs)
                pr_aucs.append(auc(recall, precision))
            return np.mean(pr_aucs)

        print("Running Optuna study (50 trials)...")
        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=50)
        best_params = study.best_params

        mlflow.log_params({f"optuna_{k}": v for k, v in best_params.items()})
        mlflow.log_metric("optuna_best_cv_pr_auc", study.best_value)

        final_params = {
            **best_params,
            'scale_pos_weight': (1 - DATA_CONFIG["fraud_rate"]) / DATA_CONFIG["fraud_rate"],
            'random_state': DATA_CONFIG["random_state"]
        }
        final_model = xgb.XGBClassifier(**final_params)
        final_model.fit(X_train_scaled, y_train)

        test_probs = final_model.predict_proba(X_test_scaled)[:, 1]
        precision, recall, _ = precision_recall_curve(y_test, test_probs)
        test_pr_auc = auc(recall, precision)
        test_preds = (test_probs >= 0.5).astype(int)
        test_f1 = f1_score(y_test, test_preds)

        mlflow.log_metrics({"test_pr_auc": test_pr_auc, "test_f1": test_f1})
        mlflow.xgboost.log_model(final_model, artifact_path="xgb_model")
        print(f"Run complete! Test PR-AUC: {test_pr_auc:.4f}, Test F1: {test_f1:.4f}")

if __name__ == "__main__":
    main()
''')

!python scripts/train_with_tracking.py

2026/08/05 09:50:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/05 09:50:11 INFO mlflow.store.db.utils: Updating database tables
2026/08/05 09:50:14 INFO mlflow.tracking.fluent: Experiment with name 'payment-fraud-detection' does not exist. Creating a new experiment.
Started MLflow Run ID: 3d852718d5974239860a440a28af8923
Running Optuna study (50 trials)...
2026/08/05 09:52:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Run complete! Test PR-AUC: 0.6314, Test F1: 0.6216


In [3]:
import os
import sys
import time
import socket
import subprocess
from pyngrok import ngrok

# 1. Kill any existing MLflow or ngrok background processes
os.system("pkill -f mlflow")
os.system("pkill -f ngrok")
time.sleep(2)

# 2. Add your ngrok authtoken
!ngrok config add-authtoken 3HSNG2N2RqLVqQd1CnFTNoe3ey0_7HvE1c5Shqz1ckDJhMbtt

# 3. Launch MLflow server robustly using Python module execution
cmd = [sys.executable, "-m", "mlflow", "server", "--host", "127.0.0.1", "--port", "5000"]
mlflow_process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# 4. Wait for port 5000 to actually open and accept connections
print("Starting MLflow server...")
for _ in range(10):
    time.sleep(1)
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", 5000)) == 0:
            print("✅ MLflow server is active on http://127.0.0.1:5000!")
            break
else:
    _, stderr = mlflow_process.communicate()
    print("❌ MLflow server failed to start:")
    print(stderr.decode())

# 5. Connect ngrok tunnel
public_url = ngrok.connect("127.0.0.1:5000")
print(f"\n👉 MLflow UI Dashboard: {public_url}")

ModuleNotFoundError: No module named 'pyngrok'